In [1]:
import matplotlib.pyplot as plt
import numpy as np
import deeplenstronomy.deeplenstronomy as dl
from deeplenstronomy.visualize import view_image
import torch
from torch.utils.data import Dataset, random_split
from torch.utils.data import DataLoader
import torch.nn as nn
import torchvision.models as models

import json
import glob
import os
 
import pandas as pd
from astropy.io import fits

/Users/2014kz/Documents/PythonStuff/dspl/test/lib/python3.10/site-packages/numba/core/decorators.py:262: NumbaDeprecationWarning: numba.generated_jit is deprecated. Please see the documentation at: https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-generated-jit for more information and advice on a suitable replacement.
  warnings.warn(msg, NumbaDeprecationWarning)


In [2]:
class LensDataset(Dataset):
    def __init__(self, images, targets, transform=None):
        self.images = images
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.images[idx]).float()
        target = torch.tensor([self.targets[idx]], dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
        return img, target

In [3]:
def asinh_stretch(img, scale=0.1):
    return np.arcsinh(img / scale)

def asinh_stretch_adaptive(img, k=3.0):
    sigma = np.median(np.abs(img - np.median(img))) * 1.4826  # robust MAD-based sigma
    scale = k * sigma
    return np.arcsinh(img / (scale + 1e-12))
 
 
def normalize_image(img):
    return (img - img.mean()) / (img.std() + 1e-8)


def preprocess_images(images):
    out = np.empty_like(images, dtype=np.float32)
    for i in range(images.shape[0]):
        stretched = asinh_stretch_adaptive(images[i])
        out[i] = normalize_image(stretched)
    return out

In [4]:
config_file = 'test.yaml'
dataset = dl.make_dataset(config_file)

img = dataset.CONFIGURATION_1_images[0]  # shape: (n_bands, numPix, numPix)

In [5]:
images = np.array(dataset.CONFIGURATION_1_images)
metadata = dataset.CONFIGURATION_1_metadata

TARGET_COL = 'PLANE_1-OBJECT_1-MASS_PROFILE_1-theta_E-I_E'

theta_values = metadata[TARGET_COL].values.astype(np.float32)
images = preprocess_images(images)

In [6]:
n_total = len(images)
n_train = int(0.7 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

generator = torch.Generator().manual_seed(42)
train_idx, val_idx, test_idx = random_split(
    range(n_total), [n_train, n_val, n_test], generator=generator
)

# normalization stats from TRAIN split only
e1_mean = theta_values[train_idx.indices].mean()
e1_std = theta_values[train_idx.indices].std()
e1_norm = (theta_values - e1_mean) / e1_std

# separate dataset instances so the train-only transform can't leak into val/test (Bug 2)
train_dataset = LensDataset(images, e1_norm, transform=None)
eval_dataset = LensDataset(images, e1_norm, transform=None)

train_set = torch.utils.data.Subset(train_dataset, train_idx.indices)
val_set = torch.utils.data.Subset(eval_dataset, val_idx.indices)
test_set = torch.utils.data.Subset(eval_dataset, test_idx.indices)

BATCH_SIZE = 32
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

dataset.CONFIGURATION_1_metadata['PLANE_1-OBJECT_1-MASS_PROFILE_1-theta_E-I_E']
TARGET_COL ='PLANE_1-OBJECT_1-MASS_PROFILE_1-theta_E-I_E'


e1_values = metadata[TARGET_COL].values.astype(np.float32)

# split indices first (see Bug 2 fix), then:
e1_mean = e1_values[train_idx.indices].mean()
e1_std = e1_values[train_idx.indices].std()
e1_norm = (e1_values - e1_mean) / e1_std

train_dataset = LensDataset(images, e1_norm, transform=None)
eval_dataset = LensDataset(images, e1_norm, transform=None)

In [7]:
class ResBlock(nn.Module):
    """
    Basic pre-activation residual block with GroupNorm.
    GroupNorm (rather than BatchNorm) is used because it's stable at small
    batch sizes and doesn't depend on batch statistics -- useful here since
    lens cutouts are often trained with modest batch sizes (e.g. 32).
    """
    def __init__(self, channels, groups=8):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, channels)
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(groups, channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.act = nn.SiLU(inplace=True)
 
    def forward(self, x):
        identity = x
        out = self.conv1(self.act(self.norm1(x)))
        out = self.conv2(self.act(self.norm2(out)))
        return out + identity
 
 
class DownBlock(nn.Module):
    """Strided conv downsample + N residual blocks at the new resolution."""
    def __init__(self, in_ch, out_ch, n_blocks=2, groups=8):
        super().__init__()
        self.downsample = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=2, padding=1)
        self.blocks = nn.ModuleList([ResBlock(out_ch, groups=groups) for _ in range(n_blocks)])
 
    def forward(self, x):
        x = self.downsample(x)
        for block in self.blocks:
            x = block(x)
        return x
 
 
class SEBlock(nn.Module):
    """
    Squeeze-and-excitation channel attention. Cheap (few params) and helps
    the network weight informative channels (e.g. ring/arc-sensitive
    features) more than noise-dominated ones.
    """
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 4)),
            nn.SiLU(inplace=True),
            nn.Linear(max(channels // reduction, 4), channels),
            nn.Sigmoid(),
        )
 
    def forward(self, x):
        b, c, _, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1, 1)
        return x * s
 
 
class LensRegressor(nn.Module):
    def __init__(self, n_bands=1, base_channels=16, groups=8, dropout=0.3):
        super().__init__()
 
        # Stem: light conv to get to base_channels before downsampling starts
        self.stem = nn.Sequential(
            nn.Conv2d(n_bands, base_channels, kernel_size=5, padding=2, bias=False),
            nn.GroupNorm(groups, base_channels),
            nn.SiLU(inplace=True),
        )
 
        c1, c2, c3, c4 = base_channels, base_channels * 2, base_channels * 4, base_channels * 8
 
        self.stage1 = DownBlock(c1, c1, n_blocks=2, groups=groups)   # 96 -> 48
        self.stage2 = DownBlock(c1, c2, n_blocks=2, groups=groups)   # 48 -> 24
        self.stage3 = DownBlock(c2, c3, n_blocks=2, groups=groups)   # 24 -> 12
        self.stage4 = DownBlock(c3, c4, n_blocks=2, groups=groups)   # 12 -> 6
 
        self.se = SEBlock(c4)
 
        self.pool = nn.AdaptiveAvgPool2d(1)
 
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(c4, 64),
            nn.SiLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.SiLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(32, 1),
        )
 
        self._init_weights()
 
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)
 
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.se(x)
        x = self.pool(x)
        return self.head(x)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_bands = images.shape[1]
 
model = LensRegressor(n_bands=n_bands).to(device)
 
with torch.no_grad():
    x_check, y_check = next(iter(train_loader))
    out_check = model(x_check.to(device))
print("Untrained output stats:", out_check.mean().item(), out_check.std().item())

Untrained output stats: -0.374595582485199 0.8250660300254822


In [9]:
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_bands = images.shape[1]
 
model = LensRegressor(n_bands=n_bands).to(device)
criterion = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
 
 
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    with torch.set_grad_enabled(is_train):
        for images_b, targets_b in loader:
            images_b, targets_b = images_b.to(device), targets_b.to(device)
            if is_train:
                optimizer.zero_grad()
            pred_mean = model(images_b)
            loss = criterion(pred_mean, targets_b)
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images_b.size(0)
    return total_loss / len(loader.dataset)
 
 
N_EPOCHS = 400
PATIENCE = 10
best_val_loss = float("inf")
epochs_no_improve = 0
history = {"train_loss": [], "val_loss": []}

for epoch in range(N_EPOCHS):
    train_loss = run_epoch(model, train_loader, optimizer)
    val_loss = run_epoch(model, val_loader)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"Epoch {epoch:03d} | train={train_loss:.4f} | val={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_lens_theta_e_model.pt")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (best val={best_val_loss:.4f})")
            break

model.load_state_dict(torch.load("best_lens_theta_e_model.pt"))

Epoch 000 | train=0.6028 | val=0.0697
Epoch 001 | train=0.0919 | val=0.0330
Epoch 002 | train=0.0731 | val=0.0204
Epoch 003 | train=0.0639 | val=0.0218
Epoch 004 | train=0.0553 | val=0.0145
Epoch 005 | train=0.0500 | val=0.0431
Epoch 006 | train=0.0461 | val=0.0128
Epoch 007 | train=0.0458 | val=0.0251
Epoch 008 | train=0.0437 | val=0.0132
Epoch 009 | train=0.0381 | val=0.0177
Epoch 010 | train=0.0358 | val=0.0129
Epoch 011 | train=0.0339 | val=0.0116
Epoch 012 | train=0.0356 | val=0.0101
Epoch 013 | train=0.0328 | val=0.0332
Epoch 014 | train=0.0321 | val=0.0120
Epoch 015 | train=0.0324 | val=0.0118
Epoch 016 | train=0.0297 | val=0.0079
Epoch 017 | train=0.0277 | val=0.0106
Epoch 018 | train=0.0270 | val=0.0102
Epoch 019 | train=0.0255 | val=0.0121
Epoch 020 | train=0.0261 | val=0.0083
Epoch 021 | train=0.0273 | val=0.0125
Epoch 022 | train=0.0241 | val=0.0177
Epoch 023 | train=0.0212 | val=0.0081
Epoch 024 | train=0.0199 | val=0.0073
Epoch 025 | train=0.0207 | val=0.0087
Epoch 026 | 

<All keys matched successfully>

In [11]:
LENS_ROOT = "./lens"
CHECKPOINT_PATH = "best_lens_theta_e_model.pt"
TARGET_NUMPIX = 96
STRETCH_SCALE = 0.1

T_MEAN = 1.139732
T_STD = 0.449532

def center_crop(img, target_numpix):
    h, w = img.shape[-2:]
    start_y = (h - target_numpix) // 2
    start_x = (w - target_numpix) // 2
    if h < target_numpix or w < target_numpix:
        raise ValueError(f"Image too small to crop: {h}x{w} < {target_numpix}")
    return img[..., start_y:start_y + target_numpix, start_x:start_x + target_numpix]


def preprocess(img, stretch_scale=STRETCH_SCALE):
    img = np.arcsinh(img / stretch_scale)
    img = (img - img.mean()) / (img.std() + 1e-8)
    return img

def extract_true_theta_e(result_json):
    # Try a few common shapes
    if "einstein_radius_effective_median_pdf" in result_json:
        return float(result_json["einstein_radius_effective_median_pdf"])
    if "kwargs_lens" in result_json:
        # e.g. list of dicts, first one being the main deflector
        return float(result_json["kwargs_lens"][0]["theta_E"])
    if "lens_mass" in result_json and "theta_E" in result_json["lens_mass"]:
        return float(result_json["lens_mass"]["theta_E"])
    raise KeyError(
        f"Could not find theta_E in JSON keys: {list(result_json.keys())}. "
        "Update extract_true_theta_e() to match your schema."
    )


def find_lens_dirs(root):
    return sorted(
        d for d in glob.glob(os.path.join(root, "*"))
        if os.path.isdir(d)
    )

def find_fits_and_json(lens_dir):
    fits_files = glob.glob(os.path.join(lens_dir, "*.fits"))
    json_files = glob.glob(os.path.join(lens_dir, "result_lens_mass.json"))
    if not fits_files or not json_files:
        return None, None
    return fits_files[0], json_files[0]


model = LensRegressor()
model.load_state_dict(torch.load('best_lens_theta_e_model.pt'))
model.eval()


records = []
lens_dirs = find_lens_dirs(LENS_ROOT)
print(f"Found {len(lens_dirs)} lens directories under {LENS_ROOT}")
 
for lens_dir in lens_dirs:
    lens_id = os.path.basename(lens_dir)
    fits_path, json_path = find_fits_and_json(lens_dir)
 
    if fits_path is None:
        print(f"[skip] {lens_id}: missing .fits or result_lens_mass.json")
        continue
 
    try:
        def load_image_data(fits_path):
            with fits.open(fits_path) as hdul:
                for hdu in hdul:
                    if hdu.data is not None and hdu.data.ndim >= 2:
                        return hdu.data.astype(np.float32)
            raise ValueError(f"No 2D image data found in any HDU of {fits_path}")
 
        img = load_image_data(fits_path)
        img = center_crop(img, TARGET_NUMPIX)
        img = preprocess(img)
        img_tensor = torch.from_numpy(img[np.newaxis, np.newaxis, ...]).float()
 
        with torch.no_grad():
            pred_norm = model(img_tensor).cpu().item()
        pred_theta_e = pred_norm * T_STD + T_MEAN
 
        with open(json_path) as f:
            result = json.load(f)
        true_theta_e = extract_true_theta_e(result)
        true_theta_e_lo = float(result.get("einstein_radius_effective_lower_1_sigma", np.nan))
        true_theta_e_hi = float(result.get("einstein_radius_effective_upper_1_sigma", np.nan))
        true_theta_e_err_lo = true_theta_e - true_theta_e_lo
        true_theta_e_err_hi = true_theta_e_hi - true_theta_e
 
        records.append({
            "lens_id": lens_id,
            "true_theta_e": true_theta_e,
            "true_theta_e_lo": true_theta_e_lo,
            "true_theta_e_hi": true_theta_e_hi,
            "true_theta_e_err_lo": true_theta_e_err_lo,
            "true_theta_e_err_hi": true_theta_e_err_hi,
            "pred_theta_e": pred_theta_e,
            "residual": pred_theta_e - true_theta_e,
            "abs_error": abs(pred_theta_e - true_theta_e),
        })

    except Exception as e:
        print(f"[error] {lens_id}: {e}")
        continue

Found 335 lens directories under ./lens
[skip] 102019125_NEG586359480506090347: missing .fits or result_lens_mass.json
[skip] 102020528_NEG557594873487530963: missing .fits or result_lens_mass.json
[skip] 102020528_NEG557780503491374019: missing .fits or result_lens_mass.json
[skip] 102020530_NEG572378924491097026: missing .fits or result_lens_mass.json
[skip] 102021017_NEG634146279486865014: missing .fits or result_lens_mass.json
[skip] 102021988_NEG637841878475955814: missing .fits or result_lens_mass.json
[skip] 102022975_NEG617890913466103850: missing .fits or result_lens_mass.json
[skip] 102045468_NEG535850797268396974: missing .fits or result_lens_mass.json
[skip] 102158586_2707908112649845702: missing .fits or result_lens_mass.json
[skip] 102158589_2744607815651760624: missing .fits or result_lens_mass.json
[skip] 102159481_2635212626663669301: missing .fits or result_lens_mass.json
[skip] 102159483_2659872799666371332: missing .fits or result_lens_mass.json
[skip] 102160056_266

In [12]:
df = pd.DataFrame(records)
 
if df.empty:
    raise RuntimeError("No lenses were successfully processed. Check paths/schema.")
 
df.to_csv("theta_e_comparison.csv", index=False)
print(f"\nSaved {len(df)} results to theta_e_comparison.csv")
 
rmse = np.sqrt((df["residual"] ** 2).mean())
mae = df["abs_error"].mean()
bias = df["residual"].mean()
 
print(f"\nN = {len(df)}")
print(f"RMSE = {rmse:.4f}")
print(f"MAE  = {mae:.4f}")
print(f"Mean bias (pred - true) = {bias:.4f}")
 
print("\nWorst 5 predictions:")
print(df.sort_values("abs_error", ascending=False).head(5).to_string(index=False))


within_1sigma = (
    (df["pred_theta_e"] >= df["true_theta_e_lo"]) &
    (df["pred_theta_e"] <= df["true_theta_e_hi"])
)
print(f"Predictions within reported 1-sigma: {within_1sigma.mean()*100:.1f}%")


Saved 322 results to theta_e_comparison.csv

N = 322
RMSE = 0.6629
MAE  = 0.5024
Mean bias (pred - true) = 0.2258

Worst 5 predictions:
                        lens_id  true_theta_e  true_theta_e_lo  true_theta_e_hi  true_theta_e_err_lo  true_theta_e_err_hi  pred_theta_e  residual  abs_error
  102157957_2711813162637737494      4.308936         4.183598         4.426237             0.125338             0.117301      0.657034 -3.651902   3.651902
102021016_NEG629685198482852633      2.436346         2.436003         2.436873             0.000342             0.000527      0.783016 -1.653330   1.653330
  102158892_2704470582654415692      2.468371         2.465512         2.472081             0.002859             0.003710      0.911095 -1.557276   1.557276
  102157953_2669397021640472384      2.107754         2.100371         2.124202             0.007383             0.016448      0.620364 -1.487390   1.487390
  102160611_2740328687682808789      2.524841         2.523979         2.52550

In [ ]:
model.eval()
pred_means, trues = [], []
with torch.no_grad():
    for images_b, targets_b in test_loader:
        images_b = images_b.to(device)
        pred_mean = model(images_b)          # single output, no logvar
        pred_means.append(pred_mean.cpu().numpy())
        trues.append(targets_b.numpy())

pred_means = np.concatenate(pred_means).flatten() * e1_std + e1_mean
trues = np.concatenate(trues).flatten() * e1_std + e1_mean

rmse = np.sqrt(np.mean((pred_means - trues) ** 2))
mae = np.mean(np.abs(pred_means - trues))
print(f"\nTest RMSE: {rmse:.4f}")
print(f"Test MAE:  {mae:.4f}")


fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE loss")
axes[0].legend()
axes[0].set_title("Training curve")

axes[1].scatter(trues, pred_means, alpha=0.3)
lims = [trues.min(), trues.max()]
axes[1].plot(lims, lims, "r--")
axes[1].set_xlabel("True e1")
axes[1].set_ylabel("Predicted e1")
axes[1].set_title("Predicted vs. true")

residuals = pred_means - trues
axes[2].scatter(trues, residuals, alpha=0.3)
axes[2].axhline(0, color="r", linestyle="--")
axes[2].set_xlabel("True e1")
axes[2].set_ylabel("Residual (pred - true)")
axes[2].set_title("Residuals vs. true value")

plt.tight_layout()
plt.savefig("e1_regression_diagnostics.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
 
axes[0].scatter(df["true_theta_e"], df["pred_theta_e"], alpha=0.6)
lims = [
    min(df["true_theta_e"].min(), df["pred_theta_e"].min()),
    max(df["true_theta_e"].max(), df["pred_theta_e"].max()),
]
axes[0].plot(lims, lims, "r--", label="y = x")
axes[0].set_xlabel("True theta_E")
axes[0].set_ylabel("Predicted theta_E")
axes[0].set_title(f"Predicted vs. true (N={len(df)})")
axes[0].legend()
 
axes[1].scatter(df["true_theta_e"], df["residual"], alpha=0.6)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_xlabel("True theta_E")
axes[1].set_ylabel("Residual (pred - true)")
axes[1].set_title(f"Residuals (RMSE={rmse:.3f}, bias={bias:.3f})")
 
plt.tight_layout()
plt.savefig("theta_e_comparison_diagnostics.png", dpi=150)
plt.show()

In [ ]:
import scipy.stats as st

slope, intercept, r, p, se = st.linregress(df['true_theta_e'], df['pred_theta_e'])
print(f"slope={slope:.3f}, intercept={intercept:.3f}, r²={r**2:.3f}")

# How to read this:
# slope ~1, intercept ~0, r² high  -> model tracks true theta_E well
# slope << 1 (e.g. <0.3), r² low   -> model output barely responds to
#                                     true value -> regression-to-mean /
#                                     domain collapse (what we suspect here)
print(f"Predicted range: {df['pred_theta_e'].min():.3f} - {df['pred_theta_e'].max():.3f}")
print(f"True range:      {df['true_theta_e'].min():.3f} - {df['true_theta_e'].max():.3f}")
print(f"Predicted std: {df['pred_theta_e'].std():.3f}")
print(f"True std:      {df['true_theta_e'].std():.3f}")


In [ ]:
"""
Domain-gap diagnostic: compare a real Euclid FITS cutout against what the
model actually saw during training (simulated deeplenstronomy images).

Run this BEFORE trying any fix (retraining, fine-tuning, augmentation) --
it should tell us in ~10 seconds whether the problem is pixel scale,
image stretch/normalization, or something else structural.
"""

import glob
import os
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt


LENS_ROOT = "./lens"
TARGET_NUMPIX = 96
STRETCH_SCALE = 0.1

# The pixel scale your deeplenstronomy config used during training
TRAINING_PIXEL_SCALE = 0.1  # arcsec/pixel, from IMAGE.PARAMETERS.pixel_scale


def load_image_data(fits_path):
    with fits.open(fits_path) as hdul:
        for hdu in hdul:
            if hdu.data is not None and hdu.data.ndim >= 2:
                return hdu.data.astype(np.float32), hdu.header
    raise ValueError(f"No 2D image data found in any HDU of {fits_path}")


def get_pixel_scale_from_header(header):
    """
    Try the common WCS keywords for pixel scale, in arcsec/pixel.
    Different pipelines use different conventions -- check several.
    """
    candidates = {}

    if "CDELT1" in header:
        # CDELT is usually in degrees/pixel -> convert to arcsec
        candidates["CDELT1 (deg->arcsec)"] = abs(header["CDELT1"]) * 3600.0
    if "CD1_1" in header:
        candidates["CD1_1 (deg->arcsec)"] = abs(header["CD1_1"]) * 3600.0
    if "PIXSCALE" in header:
        candidates["PIXSCALE (as-is)"] = header["PIXSCALE"]
    if "PIXELSCL" in header:
        candidates["PIXELSCL (as-is)"] = header["PIXELSCL"]
    if "SECPIX" in header:
        candidates["SECPIX (as-is)"] = header["SECPIX"]

    return candidates


def center_crop(img, target_numpix):
    h, w = img.shape[-2:]
    start_y = (h - target_numpix) // 2
    start_x = (w - target_numpix) // 2
    return img[..., start_y:start_y + target_numpix, start_x:start_x + target_numpix]


def preprocess(img, stretch_scale=STRETCH_SCALE):
    img = np.arcsinh(img / stretch_scale)
    img = (img - img.mean()) / (img.std() + 1e-8)
    return img


# --------------------------------------------------------------------------
# Grab one real lens and inspect it
# --------------------------------------------------------------------------
lens_dirs = sorted(d for d in glob.glob(os.path.join(LENS_ROOT, "*")) if os.path.isdir(d))
if not lens_dirs:
    raise RuntimeError(f"No lens directories found under {LENS_ROOT}")

sample_dir = lens_dirs[0]
fits_files = glob.glob(os.path.join(sample_dir, "*.fits"))
if not fits_files:
    raise RuntimeError(f"No FITS file found in {sample_dir}")

fits_path = fits_files[0]
print(f"Inspecting: {fits_path}\n")

raw_img, header = load_image_data(fits_path)

print("=== HEADER / PIXEL SCALE CHECK ===")
scale_candidates = get_pixel_scale_from_header(header)
if not scale_candidates:
    print("No standard pixel-scale keyword found in header.")
    print("Full header dump below -- look for scale/WCS info manually:\n")
    print(repr(header))
else:
    for key, val in scale_candidates.items():
        match = "  <-- MATCHES training" if abs(val - TRAINING_PIXEL_SCALE) < 0.01 else "  <-- MISMATCH vs training"
        print(f"  {key}: {val:.5f} arcsec/pixel{match}")
print(f"\nTraining pixel scale (from deeplenstronomy config): {TRAINING_PIXEL_SCALE} arcsec/pixel")

print("\n=== RAW IMAGE STATS ===")
print(f"Shape: {raw_img.shape}")
print(f"min={raw_img.min():.4g}  max={raw_img.max():.4g}  "
      f"mean={raw_img.mean():.4g}  std={raw_img.std():.4g}")
print(f"Fraction of pixels <= 0: {(raw_img <= 0).mean()*100:.2f}%")
print(f"Fraction of pixels that are NaN/inf: "
      f"{(~np.isfinite(raw_img)).mean()*100:.2f}%")

# --------------------------------------------------------------------------
# Side-by-side visual: raw, arcsinh-stretched, normalized
# --------------------------------------------------------------------------
cropped = center_crop(raw_img, TARGET_NUMPIX)
stretched = np.arcsinh(cropped / STRETCH_SCALE)
normalized = preprocess(cropped)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
im0 = axes[0].imshow(cropped, cmap="viridis")
axes[0].set_title("Raw (cropped)")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(stretched, cmap="viridis")
axes[1].set_title("arcsinh stretched")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(normalized, cmap="viridis")
axes[2].set_title("Normalized (model input)")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.savefig("real_lens_diagnostic.png", dpi=150)
plt.show()

print("\nSaved visualization to real_lens_diagnostic.png")
print("\n=== WHAT TO LOOK FOR ===")
print("1. If the pixel-scale check above shows a MISMATCH, that's very")
print("   likely the primary bug: the apparent angular size of any ring")
print("   in this image will be wrong relative to what the model learned.")
print("2. In the 'Normalized (model input)' panel, does a ring/arc structure")
print("   look visually similar in scale/shape to your earlier simulated")
print("   examples? If it looks much smaller/larger/blurrier, that's the")
print("   domain gap made visible.")
print("3. Check the 'Fraction of pixels <= 0' number -- arcsinh(negative/scale)")
print("   is still defined (odd function) but if a large fraction of pixels")
print("   are negative (common in background-subtracted real data), the")
print("   stretched image statistics can look very different from sim data,")
print("   which is rarely background-subtracted the same way.")

In [ ]:
print("Real image: std =", raw_img.std(), " std/STRETCH_SCALE =", raw_img.std()/0.1)
sim_img = dataset.CONFIGURATION_1_images[0][0] 
print("Sim image:  std =", sim_img.std(), " std/STRETCH_SCALE =", sim_img.std()/0.1)

In [ ]:
def robust_sigma(img):
    return np.median(np.abs(img - np.median(img))) * 1.4826

real_scale = 3.0 * robust_sigma(raw_img)
sim_scale = 3.0 * robust_sigma(sim_img)

print("Real: std/adaptive_scale =", raw_img.std() / real_scale)
print("Sim:  std/adaptive_scale =", sim_img.std() / sim_scale)

In [ ]:
def light_extent_snr(img, threshold_sigma=3):
    bg = np.median(img)
    noise = np.median(np.abs(img - bg)) * 1.4826
    frac_above_thresh = (img > bg + threshold_sigma * noise).mean()
    return frac_above_thresh

print("Real: fraction of pixels >3sigma above bg:", light_extent_snr(raw_img))
print("Sim:  fraction of pixels >3sigma above bg:", light_extent_snr(sim_img))